# Preprocessing Dataset Subride

Notebook ini dibuat ulang dari `dataset/subride.csv`. Fokusnya cuma membersihkan teks review, menjaga label, lalu membuat data yang siap dipakai untuk TF-IDF dan model klasifikasi.


In [ ]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.utils import resample


In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "dataset" / "subride.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "dataset" / "subride.csv"
OUTPUT_PATH = PROJECT_ROOT / "dataset" / "subride_clean_rapi.csv"
PLOT_PATH = PROJECT_ROOT / "model" / "preprocessing_summary_rapi.png"

DATA_PATH


## 1. Baca data awal

Bagian ini cuma buat cek bentuk data, nama kolom, dan pembagian label sebelum dibersihkan.


In [ ]:
df_awal = pd.read_csv(DATA_PATH)

print("Jumlah baris dan kolom:", df_awal.shape)
print("Kolom:", list(df_awal.columns))
print("\nJumlah label:")
print(df_awal["label"].value_counts().sort_index())

df_awal.head()


## 2. Ambil kolom yang dipakai

Model nanti cuma butuh teks review dan label. Kolom seperti score, app, dan translated_review tidak dibuang dari file asli, tapi tidak dipakai di data latih ini.


In [ ]:
df = df_awal[["review", "label"]].copy()

print("Data kosong per kolom:")
print(df.isna().sum())

print("\nDuplikat review sebelum dibersihkan:", df.duplicated(subset=["review"]).sum())
df.head()


## 3. Normalisasi teks

Di sini teks dibuat lowercase, link dan simbol dibersihkan, singkatan umum diseragamkan, lalu spasi dirapikan. Kamusnya sengaja tidak terlalu agresif supaya makna asli review tidak banyak berubah.


In [ ]:
kamus_normalisasi = {
    "yg": "yang",
    "yng": "yang",
    "sy": "saya",
    "sya": "saya",
    "gw": "saya",
    "gue": "saya",
    "aku": "saya",
    "lo": "kamu",
    "lu": "kamu",
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "engga": "tidak",
    "enggak": "tidak",
    "tdk": "tidak",
    "tak": "tidak",
    "gada": "tidak ada",
    "gaada": "tidak ada",
    "gabisa": "tidak bisa",
    "tp": "tapi",
    "tpi": "tapi",
    "jg": "juga",
    "jgn": "jangan",
    "krn": "karena",
    "karna": "karena",
    "klo": "kalau",
    "kl": "kalau",
    "kalo": "kalau",
    "klw": "kalau",
    "bgt": "banget",
    "bngt": "banget",
    "sgt": "sangat",
    "pake": "pakai",
    "pke": "pakai",
    "pesen": "pesan",
    "dapet": "dapat",
    "dpt": "dapat",
    "ngambil": "mengambil",
    "nyari": "mencari",
    "nunggu": "menunggu",
    "nungguin": "menunggu",
    "bener": "benar",
    "deket": "dekat",
    "emang": "memang",
    "kayak": "seperti",
    "kyk": "seperti",
    "sampe": "sampai",
    "ampe": "sampai",
    "tau": "tahu",
    "nih": "ini",
    "tuh": "itu",
    "apk": "aplikasi",
    "apli": "aplikasi",
    "lbh": "lebih",
    "sdh": "sudah",
    "udah": "sudah",
    "udh": "sudah",
    "blm": "belum",
    "lg": "lagi",
    "dl": "dulu",
    "dlu": "dulu",
    "skrg": "sekarang",
    "skrang": "sekarang",
    "nnti": "nanti",
    "ntr": "nanti",
    "dr": "dari",
    "dri": "dari",
    "d": "di",
    "dg": "dengan",
    "dgn": "dengan",
    "pd": "pada",
    "byk": "banyak",
    "bnyk": "banyak",
    "msh": "masih",
    "msih": "masih",
    "krg": "kurang",
    "gmn": "bagaimana",
    "gimana": "bagaimana",
    "knp": "kenapa",
    "knpa": "kenapa",
    "utk": "untuk",
    "tuk": "untuk",
    "bs": "bisa",
    "org": "orang",
    "sm": "sama",
    "aja": "saja",
    "bln": "bulan",
    "thn": "tahun",
    "mnt": "menit",
    "pdhl": "padahal",
    "jmpt": "jemput",
    "dtg": "datang",
    "bkn": "bukan",
    "gitu": "begitu",
    "trs": "terus",
    "trus": "terus",
    "tlg": "tolong",
    "plis": "tolong",
    "mnta": "minta",
    "smg": "semoga",
    "ckp": "cukup",
    "hrus": "harus",
    "cmn": "cuma",
    "cman": "cuma",
    "ok": "oke",
    "makasih": "terima kasih",
    "mksh": "terima kasih",
    "mksih": "terima kasih",
    "trims": "terima kasih",
    "trimakasih": "terima kasih",
    "terimakasih": "terima kasih",
    "donk": "dong",
    "tlp": "telepon",
    "telfon": "telepon",
    "rmh": "rumah",
    "k": "ke",
    "maxsim": "maxim",
    "maxin": "maxim",
    "gojk": "gojek",
    "grb": "grab",
    "ojol": "ojek online",
    "goride": "gojek",
    "ongkir": "ongkos kirim",
    "dicancel": "dibatalkan",
    "cancle": "cancel",
    "ngecancel": "membatalkan",
    "ngancel": "membatalkan",
    "gajalan": "tidak jalan",
    "gajelas": "tidak jelas",
    "gjls": "tidak jelas",
    "diem": "diam",
    "nganter": "mengantar",
    "mantep": "mantap",
    "mantul": "mantap",
}


In [ ]:
def rapikan_spasi(teks):
    return re.sub(r"\s+", " ", teks).strip()


def normalisasi_teks(teks):
    if pd.isna(teks):
        return ""

    teks = str(teks).lower()
    teks = re.sub(r"https?://\S+|www\.\S+", " ", teks)
    teks = re.sub(r"\S+@\S+", " ", teks)
    teks = re.sub(r"([a-z])\1{2,}", r"\1", teks)

    teks = re.sub(r"(\d+)\s*rb\b", r"\1 ribu", teks)
    teks = re.sub(r"(\d+)\s*jt\b", r"\1 juta", teks)
    teks = re.sub(r"(\d+)\s*k\b", r"\1 ribu", teks)
    teks = re.sub(r"(\d+)\s*x\b", r"\1 kali", teks)
    teks = re.sub(r"(\d+)\s*mnt\b", r"\1 menit", teks)
    teks = re.sub(r"([a-z]+)(\d+)", r"\1 \2", teks)
    teks = re.sub(r"(\d+)([a-z]+)", r"\1 \2", teks)
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)
    teks = rapikan_spasi(teks)

    hasil = []
    for kata in teks.split():
        kata_baru = kamus_normalisasi.get(kata, kata)
        hasil.extend(kata_baru.split())

    return rapikan_spasi(" ".join(hasil))


In [ ]:
contoh = df["review"].head(5).copy()
contoh_hasil = contoh.apply(normalisasi_teks)

pd.DataFrame({
    "sebelum": contoh,
    "sesudah": contoh_hasil,
})


## 4. Bersihkan data

Setelah normalisasi, baris kosong dan review duplikat dibuang. Ini dilakukan setelah normalisasi karena dua review yang awalnya beda penulisan bisa saja menjadi sama.


In [ ]:
jumlah_awal = len(df)

df = df.dropna(subset=["review", "label"]).copy()
df["label"] = df["label"].astype(int)
df["review"] = df["review"].apply(normalisasi_teks)
df = df[df["review"].str.len() > 0].copy()
df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)

print("Jumlah awal:", jumlah_awal)
print("Jumlah setelah bersih:", len(df))
print("\nLabel setelah bersih:")
print(df["label"].value_counts().sort_index())

df.head()


## 5. Stopword dan stemming

Stopword removal tidak dipakai untuk data latih, karena kata negasi seperti `tidak`, `belum`, `jangan`, dan `gak` penting buat makna review. Stemming juga tidak dipakai di sini karena data sudah cukup pendek dan TF-IDF bisa langsung bekerja dari token hasil normalisasi.

Stopword hanya dipakai nanti untuk grafik supaya kata umum tidak mendominasi visualisasi.


## 6. Samakan jumlah label

Label di data awal tidak seimbang. Supaya model tidak terlalu berat ke kelas mayoritas, data dibuat seimbang dengan undersampling dari label yang jumlahnya lebih banyak.


In [ ]:
label_min = df["label"].value_counts().min()
bagian_label = []

for label, data_label in df.groupby("label"):
    data_sample = resample(
        data_label,
        replace=False,
        n_samples=label_min,
        random_state=42,
    )
    bagian_label.append(data_sample)

df_bersih = (
    pd.concat(bagian_label)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Jumlah final:", len(df_bersih))
print("\nLabel final:")
print(df_bersih["label"].value_counts().sort_index())

df_bersih.head()


## 7. Simpan hasil

File hasil disimpan sebagai CSV baru supaya file asli tetap aman.


In [ ]:
df_bersih.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH


## 8. Cek kata yang sering muncul

Bagian ini cuma untuk gambaran data setelah preprocessing, bukan untuk mengubah data latih.


In [ ]:
stopword_grafik = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "pada",
    "ini", "itu", "saya", "kamu", "ada", "jadi", "sama", "saja",
    "nya", "lebih", "lagi",
}

semua_kata = []
for teks in df_bersih["review"]:
    semua_kata.extend(
        kata
        for kata in teks.split()
        if kata not in stopword_grafik and len(kata) > 2
    )

top_kata = Counter(semua_kata).most_common(15)
pd.DataFrame(top_kata, columns=["kata", "jumlah"])


In [ ]:
kata, jumlah = zip(*top_kata)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(kata, jumlah, color="#2f80ed")
ax.invert_yaxis()
ax.set_title("Kata yang sering muncul setelah preprocessing")
ax.set_xlabel("Jumlah")
fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()


## 9. Ringkasan akhir

Bagian terakhir ini bisa dipakai untuk bahan jelasin hasil preprocessing saat presentasi.


In [ ]:
panjang_kata = df_bersih["review"].str.split().str.len()

print("File output:", OUTPUT_PATH)
print("File grafik:", PLOT_PATH)
print("Jumlah data final:", len(df_bersih))
print("Distribusi label final:")
print(df_bersih["label"].value_counts().sort_index())
print("\nRata-rata jumlah kata:", round(panjang_kata.mean(), 2))
print("Jumlah kata paling pendek:", panjang_kata.min())
print("Jumlah kata paling panjang:", panjang_kata.max())

df_bersih.sample(5, random_state=42)
